# Course 11 Lab — Solution Architecture and Technical Strategy

**Capstone:** should Northstar build a centralized RAG and agent platform?

We turn stakeholder concerns into measurable scenarios, compare credible options with
version-bound evidence and uncertainty, model economics, record an accountable decision, and
design a reversible migration.

## 1. Scope and decision boundary

This notebook is deterministic, credential-free and based on synthetic pilot evidence. Its
recommendation demonstrates a method; it is not a real procurement or cloud recommendation.
Scores inform judgment. The trusted review process validates evidence and records the decision;
no model, matrix or vendor claim creates authorization.

In [1]:
from dataclasses import replace

from lab import (
    DecisionDisposition,
    DecisionStatus,
    StageStatus,
    assess_migration,
    calculate_tco,
    demo_adr,
    demo_concerns,
    demo_criteria,
    demo_economics,
    demo_evidence,
    demo_migration,
    demo_options,
    demo_quality_scenarios,
    evaluate_option,
    rank_options,
    run_demo_strategy,
    sensitivity_analysis,
    validate_adr,
)

TRUSTED = frozenset({"architecture-review-office"})
NOW = 1_300
print("Course 11 architecture strategy lab ready")

Course 11 architecture strategy lab ready


## 2. Architecture decision lifecycle

In [2]:
lifecycle = [
    "outcomes and stakeholders", "quality scenarios and constraints", "credible options",
    "evidence, economics and risk", "trade-off and sensitivity", "ADR/RFC",
    "transition stages", "runtime feedback and reversal",
]
for index, stage in enumerate(lifecycle, start=1):
    print(index, stage)

1 outcomes and stakeholders
2 quality scenarios and constraints
3 credible options
4 evidence, economics and risk
5 trade-off and sensitivity
6 ADR/RFC
7 transition stages
8 runtime feedback and reversal


Architecture is not the score or diagram. The evidence chain connects why the system exists to
how a decision will be tested and changed.

## 3. Start with stakeholders and decision rights

In [3]:
concerns = demo_concerns()
for item in concerns:
    print(f"{item.stakeholder:10} | {item.outcome:35} | {item.decision_right}")

product    | faster safe use-case delivery       | prioritize outcomes
security   | bounded data and tool access        | approve controls
platform   | operable shared capabilities        | own paved road
finance    | defensible unit economics           | challenge forecast


Product, platform, security and finance own different decisions. Consultation is not approval;
review acceptance is not production deployment authority.

## 4. Make quality attributes testable

In [4]:
scenarios = demo_quality_scenarios()
for item in scenarios:
    print(item.scenario_id, "=>", item.measure, item.direction, item.threshold, item.unit)

qa-latency => p95 latency lower-is-better 2.0 seconds
qa-isolation => forbidden outcomes lower-is-better 0.0 count
qa-recovery => RTO lower-is-better 30.0 minutes


Each scenario names source, stimulus, environment, artifact, response and measure. Generic
“secure and scalable” requirements cannot drive design or validation.

## 5. Inspect the credible options

In [5]:
options = demo_options()
for option in options:
    print(
        option.option_id, option.kind, sorted(option.capabilities),
        "exit:", option.exit_strategy,
    )

point-solutions build ['local-control', 'speed'] exit: standard export contracts
central-build build ['customization', 'governance', 'shared-runtime'] exit: open APIs and artifact export
hybrid-platform hybrid ['domain-ownership', 'governance', 'managed-services', 'shared-contracts'] exit: provider adapters, open telemetry, data and prompt export


Point solutions, a fully custom central platform, and a governed hybrid each include ownership,
capability scope, provider dependencies and exit—not just product names.

## 6. Define criteria and preserve hard constraints

In [6]:
criteria = demo_criteria()
for item in criteria:
    print(
        f"{item.criterion_id:28} weight={item.weight:.2f} "
        f"hard={item.hard_threshold} {item.unit}"
    )
assert abs(sum(item.weight for item in criteria) - 1.0) < 1e-9

time-to-first-use            weight=0.12 hard=9.0 months
compliant-success            weight=0.22 hard=0.9 ratio
tenant-isolation             weight=0.18 hard=0.0 count
availability                 weight=0.12 hard=0.99 ratio
annual-tco                   weight=0.14 hard=3500000.0 usd
portability                  weight=0.10 hard=None score
team-cognitive-load          weight=0.07 hard=None score
strategic-differentiation    weight=0.05 hard=None score


Tenant isolation and compliant success require independent evidence and hard thresholds. Cost
or speed cannot compensate for a breach.

## 7. Establish the naive baseline

In [7]:
naive = {
    "point-solutions": 9, "central-build": 8, "hybrid-platform": 7,
}
print("untraceable executive preference:", max(naive, key=naive.get))
print("missing: units, constraints, evidence, confidence, cost range, reversal")

untraceable executive preference: point-solutions
missing: units, constraints, evidence, confidence, cost range, reversal


A one-number preference looks decisive but cannot be challenged or reproduced. We replace it
with exact evidence records and three decision states.

## 8. Inspect evidence contracts

In [8]:
evidence = demo_evidence()
sample = next(
    item for item in evidence
    if item.option_id == "hybrid-platform" and item.criterion_id == "compliant-success"
)
print(sample)
assert sample.independent and sample.option_version == "v1"

OptionEvidence(evidence_id='evidence:hybrid-platform:compliant-success', option_id='hybrid-platform', option_version='v1', criterion_id='compliant-success', observed_value=0.94, unit='ratio', source='northstar-pilot-2026-q3', producer='architecture-review-office', confidence=0.9, independent=True, created_at=1000, expires_at=4000)


## 9. Evaluate all options

In [9]:
decisions = tuple(
    evaluate_option(option, criteria, evidence, trusted_producers=TRUSTED, now=NOW)
    for option in options
)
for item in decisions:
    print(item.option_id, item.disposition, item.utility_score, item.reasons)

point-solutions disqualified None ('hard-threshold-failed:availability', 'hard-threshold-failed:compliant-success', 'hard-threshold-failed:tenant-isolation')
central-build disqualified None ('hard-threshold-failed:annual-tco', 'hard-threshold-failed:time-to-first-use')
hybrid-platform eligible 0.788609 ()


Point solutions fail tenant isolation. The custom central build misses time to value. The hybrid
meets hard thresholds and is eligible. Disqualification is not a low weighted score.

## 10. Rank only eligible options

In [10]:
ranking = rank_options(decisions)
print([(item.option_id, item.utility_score, item.confidence) for item in ranking])
assert [item.option_id for item in ranking] == ["hybrid-platform"]

[('hybrid-platform', 0.788609, 0.9)]


## 11. Failure injection: missing evidence

In [11]:
missing = tuple(
    item for item in evidence
    if not (item.option_id == "hybrid-platform" and item.criterion_id == "portability")
)
hybrid = next(item for item in options if item.option_id == "hybrid-platform")
unknown = evaluate_option(hybrid, criteria, missing, trusted_producers=TRUSTED, now=NOW)
print(unknown.disposition, unknown.reasons)
assert unknown.disposition is DecisionDisposition.INCONCLUSIVE

inconclusive ('missing-evidence:portability',)


Missing evidence is not an observed value of zero. The recommendation pauses until the gap is
measured or an accountable decision explicitly narrows scope.

## 12. Failure injection: vendor claim used as independent evidence

In [12]:
modified = list(evidence)
index = next(
    i for i, item in enumerate(modified)
    if item.option_id == "hybrid-platform" and item.criterion_id == "compliant-success"
)
modified[index] = replace(modified[index], producer="vendor-sales", independent=False)
untrusted = evaluate_option(hybrid, criteria, modified, trusted_producers=TRUSTED, now=NOW)
print(untrusted.disposition, untrusted.reasons)
assert untrusted.disposition is DecisionDisposition.DISQUALIFIED

disqualified ('independent-evidence-required:compliant-success', 'untrusted-evidence:compliant-success')


## 13. Failure injection: stale option version

In [13]:
wrong_version = tuple(
    replace(item, option_version="v2") if item.option_id == "hybrid-platform" else item
    for item in evidence
)
stale_version = evaluate_option(
    hybrid, criteria, wrong_version, trusted_producers=TRUSTED, now=NOW
)
print(stale_version.disposition, stale_version.reasons[:3], "…")
assert stale_version.disposition is DecisionDisposition.INCONCLUSIVE

inconclusive ('missing-evidence:annual-tco', 'missing-evidence:availability', 'missing-evidence:compliant-success') …


## 14. Inspect criterion contributions

In [14]:
hybrid_decision = next(item for item in decisions if item.option_id == "hybrid-platform")
for result in hybrid_decision.results:
    print(
        f"{result.criterion_id:28} normalized={result.normalized_score} "
        f"confidence={result.confidence} contribution={result.weighted_score}"
    )

time-to-first-use            normalized=0.7 confidence=0.9 contribution=0.0816
compliant-success            normalized=0.857143 confidence=0.9 contribution=0.184171
tenant-isolation             normalized=1.0 confidence=0.9 contribution=0.1764
availability                 normalized=0.918367 confidence=0.9 contribution=0.107804
annual-tco                   normalized=0.533333 confidence=0.9 contribution=0.071867
portability                  normalized=0.82 confidence=0.9 contribution=0.08
team-cognitive-load          normalized=0.666667 confidence=0.9 contribution=0.045267
strategic-differentiation    normalized=0.85 confidence=0.9 contribution=0.0415


Visible contributions reveal where a recommendation depends on low-confidence evidence or a
controversial weight. The score is not more precise than its assumptions.

## 15. Run sensitivity analysis

In [15]:
sensitivity = sensitivity_analysis(
    options, criteria, evidence, trusted_producers=TRUSTED, now=NOW, shift=0.15
)
print(sensitivity)
assert sensitivity.stable_winner == "hybrid-platform"

SensitivityResult(winner_counts={'point-solutions': 0, 'central-build': 0, 'hybrid-platform': 9}, tested_models=9, stable_winner='hybrid-platform')


Each criterion receives a relative 15% weight increase. The same winner survives every model;
this is evidence of local stability, not proof under every future assumption.

## 16. Model low, expected, and high total economics

In [16]:
economics = tuple(calculate_tco(item) for item in demo_economics())
for item in economics:
    print(
        item.option_id, "low/expected/high=",
        (item.low, item.expected, item.high), "unit=", item.cost_per_unit,
        "risk=", item.expected_risk_loss,
    )

point-solutions low/expected/high= (5978000.0, 7335000.0, 9439000.0) unit= 2.445 risk= 375000.0
central-build low/expected/high= (8653500.0, 10670000.0, 13915500.0) unit= 3.5567 risk= 500000.0
hybrid-platform low/expected/high= (5994000.0, 7305000.0, 9477000.0) unit= 2.435 risk= 225000.0


The model includes migration, annual platform and people, per-unit use, exit and expected risk
loss. It avoids treating subscription price or existing staff as the total cost.

## 17. Sensitivity experiment: tenfold volume

In [17]:
high_volume = tuple(
    calculate_tco(replace(model, annual_units=model.annual_units * 10))
    for model in demo_economics()
)
for item in high_volume:
    print(item.option_id, item.expected, item.cost_per_unit)

point-solutions 13275000.0 0.4425
central-build 14450000.0 0.4817
hybrid-platform 11625000.0 0.3875


Fixed cost amortizes while variable cost grows. Architecture economics must be evaluated at
realistic adoption and workload shapes rather than one average request count.

## 18. Build the architecture decision record

In [18]:
adr = demo_adr(evidence)
print(adr.adr_id, adr.status, adr.selected_option_id)
print("assumptions:", adr.assumptions)
print("reversal:", adr.reversal_triggers)
print("digest:", adr.decision_digest)

ADR-011 accepted hybrid-platform
assumptions: ('five qualified platform engineers', 'provider export remains available')
reversal: ('adoption below 50 percent after two quarters', 'unit cost above 0.35 USD', 'exit test fails')
digest: sha256:231c36c00933319029dd0764b0187c0d36a5e728211b16af1537fa2dd33057f7


## 19. Validate accountable acceptance

In [19]:
known_options = frozenset(item.option_id for item in options)
known_evidence = frozenset(item.evidence_id for item in evidence)
reasons = validate_adr(
    adr, known_options=known_options, known_evidence=known_evidence, now=NOW
)
print("ADR findings:", reasons)
assert reasons == () and adr.status is DecisionStatus.ACCEPTED

ADR findings: ()


## 20. Failure injection: alter the accepted decision

In [20]:
altered = replace(adr, decision="Centralize every domain workflow")
altered_reasons = validate_adr(
    altered, known_options=known_options, known_evidence=known_evidence, now=NOW
)
print(altered_reasons)
assert "decision-digest-mismatch" in altered_reasons

('decision-digest-mismatch',)


Acceptance binds an exact record. The owner cannot self-approve, and changing the decision,
evidence, assumptions or triggers requires a new reviewed version.

## 21. Design the transition architecture

In [21]:
migration = demo_migration()
for stage in migration:
    print(stage.stage_id, stage.status, "depends on", stage.depends_on)
assessment = assess_migration(migration)
print("ready next:", assessment.ready_stages)
assert assessment.valid and assessment.ready_stages == ("scale",)

foundation complete depends on ()
pilot complete depends on ('foundation',)
scale planned depends on ('pilot',)
consolidate planned depends on ('scale',)
ready next: ('scale',)


## 22. Failure injection: migration dependency cycle

In [22]:
cyclic = (
    replace(migration[0], depends_on=("scale",), status=StageStatus.PLANNED),
    migration[1], migration[2], migration[3],
)
cycle_result = assess_migration(cyclic)
print(cycle_result.reasons)
assert "migration-cycle" in cycle_result.reasons

('completed-before-dependency:pilot', 'migration-cycle')


A roadmap is an executable dependency and evidence model, not colored quarters. Each stage has
an owner, entry/exit evidence, rollback and kill criteria.

## 23. Compare baseline and evidence-backed practice

In [23]:
comparison = {
    "requirements": ("secure and scalable", "quality scenarios with measures"),
    "options": ("preferred vendor", "coherent build/buy/partner/hybrid systems"),
    "evidence": ("claims", "version, unit, source, producer, freshness, confidence"),
    "trade-off": ("one score", "hard gates + uncertainty + sensitivity"),
    "economics": ("price", "TCO range + risk + unit + exit"),
    "decision": ("meeting notes", "ADR with consequences and reversal"),
    "migration": ("target diagram", "staged entry/exit/rollback/kill"),
}
for topic, (before, after) in comparison.items():
    print(f"{topic:12} {before:24} -> {after}")

requirements secure and scalable      -> quality scenarios with measures
options      preferred vendor         -> coherent build/buy/partner/hybrid systems
evidence     claims                   -> version, unit, source, producer, freshness, confidence
trade-off    one score                -> hard gates + uncertainty + sensitivity
economics    price                    -> TCO range + risk + unit + exit
decision     meeting notes            -> ADR with consequences and reversal
migration    target diagram           -> staged entry/exit/rollback/kill


## 24. Execute the complete strategy path

In [24]:
result = run_demo_strategy()
print("recommended:", result["ranking"][0].option_id)
print("stable:", result["sensitivity"].stable_winner)
print("ADR findings:", result["adr_reasons"])
print("migration valid:", result["migration"].valid)
assert result["ranking"][0].option_id == "hybrid-platform"

recommended: hybrid-platform
stable: hybrid-platform
ADR findings: ()
migration valid: True


## 25. Interpretation and limitations

The hybrid recommendation follows from this synthetic dataset. A real decision needs provider
pilots, domain slices, legal/security review, capacity and failure tests, causal adoption/value
evidence, commercial terms, staffing reality, and verified exit. The method preserves where the
conclusion came from and what would change it.

## 26. Production upgrade path

| Notebook primitive | Production practice |
|---|---|
| dataclass concerns | versioned requirement/scenario registry with owners |
| synthetic observations | linked pilot, test, SLO, cost and risk evidence |
| local score | reviewed model with ranges, slices and sensitivity |
| cost function | finance-reviewed forecast versus actual unit economics |
| ADR digest | immutable decision repository and supersession workflow |
| stage tuple | portfolio/dependency tracking with release evidence |
| local review | cross-functional RFC, dissent, decision and escalation |

Automate deterministic fitness functions, but retain accountable judgment for goals, trade-offs,
ethics, legal interpretation, residual risk and organizational consequences.

## 27. Exercises

1. Add an integrated commercial-suite option with vendor and independent evidence separated.
2. Add sustainability and data-residency criteria without averaging away hard constraints.
3. Create a regulated-risk slice that fails while the aggregate succeeds.
4. Find volume, price, staffing or adoption conditions that flip the recommendation.
5. Design and validate a provider-exit stage before consolidating legacy services.

## 28. Summary

Good architecture makes reasoning inspectable: outcomes, scenarios, constraints, options,
evidence, uncertainty, costs, decisions, transitions and reversal. The recommendation matters;
the ability to test and change it matters more.